# Road Damage Detection with Ultralytics YOLO11 (Google Colab Training)

This notebook provides the complete end-to-end cloud training and evaluation pipeline for **road-damage-yolo**.

### Target Road Damage Classes:
- `D00`: Longitudinal Crack
- `D10`: Transverse Crack
- `D20`: Alligator Crack
- `D40`: Pothole

### Execution Policy:
- **Training**: Runs on Google Colab with GPU acceleration (`T4`, `A100`).
- **Credentials**: Kaggle and Roboflow credentials are retrieved securely from Google Colab Secrets (no hardcoded keys).


## 1. Check GPU Availability

Verify that a GPU runtime is active in Google Colab (Runtime -> Change runtime type -> T4 GPU).
This cell runs `nvidia-smi` to verify the driver version, GPU model, and available VRAM.

In [ ]:
# Check active GPU and CUDA driver
!nvidia-smi

## 2. Clone GitHub Repository & Install Requirements

Clone the `road-damage-yolo` repository into `/content`, navigate into the project directory, and install all required dependencies (`ultralytics`, `roboflow`, `opencv-python`, `tqdm`).
*(Idempotent: safe to run multiple times without causing nested directories)*

In [ ]:
import os
from pathlib import Path

# Replace with your GitHub repository URL if hosted remotely
REPO_URL = "https://github.com/YOUR_USERNAME/road-damage-yolo.git"
REPO_DIR = "road-damage-yolo"

# Navigate to repository root safely
if Path.cwd().name != REPO_DIR:
    target_dir = Path("/content") / REPO_DIR
    if not target_dir.exists():
        !git clone {REPO_URL} {target_dir}
    %cd {target_dir}

# Install required dependencies
!pip install -r requirements.txt

## 3. Download Road Damage Dataset

You can download your road damage training dataset directly from **Kaggle** (Recommended) or **Roboflow Universe**.

### Option A: Download from Kaggle (Recommended)
1. Go to [Kaggle](https://www.kaggle.com) -> Account Settings -> Scroll to **API** -> Click **Create New Token** to get `kaggle.json`.
2. In Google Colab, open the **Secrets** panel (key icon in the left sidebar) and add:
   - `KAGGLE_USERNAME`: Your Kaggle username
   - `KAGGLE_KEY`: Your Kaggle API key
   *(Alternatively, upload `kaggle.json` to `/root/.kaggle/kaggle.json`)*
3. Run the cell below to fetch and prepare the dataset (e.g. `aliabdelmenam/rdd-2022`).

### Option B: Download from Roboflow Universe (Alternative)
Add `ROBOFLOW_API_KEY` to Colab Secrets, configure your workspace/project/version, and run cell 3B.

In [ ]:
# =====================================================================
# 3A. Download Road Damage Dataset from Kaggle (Recommended)
# =====================================================================
import os
from google.colab import userdata

# Retrieve Kaggle credentials securely from Colab Secrets (Key icon in sidebar)
try:
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    if kaggle_token:
        os.environ['KAGGLE_API_TOKEN'] = str(kaggle_token)
        print('Successfully retrieved KAGGLE_API_TOKEN from Colab Secrets.')
    else:
        kaggle_user = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')
        if kaggle_user and kaggle_key:
            os.environ['KAGGLE_USERNAME'] = str(kaggle_user)
            os.environ['KAGGLE_KEY'] = str(kaggle_key)
            print('Successfully retrieved Kaggle credentials from Colab Secrets.')
except Exception as e:
    print(f'Notice: Could not load Kaggle secrets automatically ({e}).')
    print('Add KAGGLE_API_TOKEN in Colab Secrets tab (key icon in left sidebar).')

# Kaggle dataset slug (e.g. 'aliabdelmenam/rdd-2022' or 'chriskue/road-damage-dataset')
KAGGLE_DATASET = 'rogersinghchugh/road-damage-detection-yolo-v-11'  # Pre-structured YOLO11 RDD dataset
OUT_DIR = 'datasets/road_damage'

# Download, extract, auto-structure, and verify dataset integrity
!python src/prepare_data.py \
    --kaggle {KAGGLE_DATASET} \
    --out {OUT_DIR}

### Option B: Download from Roboflow Universe (Alternative)
If using a Roboflow dataset instead, ensure `ROBOFLOW_API_KEY` is set in Colab Secrets and run this cell.

In [ ]:
# =====================================================================
# 3B. Download Dataset from Roboflow Universe (Alternative)
# =====================================================================
import os
from google.colab import userdata

# Retrieve API key securely from Colab Secrets
roboflow_key = None
try:
    roboflow_key = userdata.get('ROBOFLOW_API_KEY')
    if roboflow_key:
        os.environ['ROBOFLOW_API_KEY'] = str(roboflow_key)
        print('Successfully retrieved ROBOFLOW_API_KEY from Colab Secrets.')
except Exception as e:
    print(f'Warning: Could not load secret ROBOFLOW_API_KEY ({e}).')

WORKSPACE = 'YOUR_WORKSPACE'
PROJECT = 'YOUR_PROJECT'
VERSION = 1
OUT_DIR = 'datasets/road_damage'

!python src/prepare_data.py \
    --workspace {WORKSPACE} \
    --project {PROJECT} \
    --version {VERSION} \
    --out {OUT_DIR}

## 4. Fine-Tune YOLO11 Model (`src/train.py`)

Fine-tune the YOLO11 small (`yolo11s.pt`) model using transfer learning on the road damage dataset.

Key Hyperparameters configured in `src/train.py`:
- `epochs=50`: Convergence across fine-tuning.
- `imgsz=640`: Input image resolution (increase to `1024` if resolving thin hairline cracks).
- `batch=16`: Sized for Tesla T4 GPU VRAM.
- `patience=15`: Early stopping to prevent overfitting.
- `flipud=0.0`: Disabled vertical flips to maintain real-world road perspective.
- `fliplr=0.5`: 50% horizontal flips for lateral crack invariance.
- `plots=True`: Generates PR curves, loss curves, and confusion matrices.

In [ ]:
# Train YOLO11 on the road damage dataset
!python src/train.py \
    --data datasets/road_damage/data.yaml \
    --model yolo11s.pt \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --project runs/train \
    --name road_damage_yolo11s

## 5. Evaluate Model on Test Split (`src/evaluate.py`)

Benchmark the fine-tuned model weights on the held-out `test` split (with automatic fallback to `val` if `test` is not defined in `data.yaml`).
This computes overall Precision, Recall, mAP@0.5, and mAP@0.5:0.95, prints a per-class breakdown for `D00, D10, D20, D40`, and exports the results to `outputs/metrics.json`.

In [ ]:
# Evaluate best model weights on test split and save outputs/metrics.json
!python src/evaluate.py \
    --weights runs/train/road_damage_yolo11s/weights/best.pt \
    --data datasets/road_damage/data.yaml \
    --split test \
    --out outputs/metrics.json

## 6. Persist Model Weights & Results (Drive Backup & Direct Download)

This cell safely saves your trained weights (`best.pt`), evaluation charts, and `metrics.json`:
1. **Option A (Google Drive)**: Attempts to mount Google Drive and back up files to `My Drive/road_damage_yolo_backup` (skips safely if canceled).
2. **Option B (Direct Download)**: Packages `best.pt`, plots, and `metrics.json` into `road_damage_results.zip` and triggers a direct browser download to your computer.

In [ ]:
import os
import shutil
import zipfile

# Locate trained checkpoint
train_run_dir = 'runs/train/road_damage_yolo11s'
best_weight = os.path.join(train_run_dir, 'weights', 'best.pt')
if not os.path.exists(best_weight) and os.path.exists('weights/best.pt'):
    best_weight = 'weights/best.pt'

# --- Option A: Attempt Google Drive Mount (Safe & Optional) ---
drive_mounted = False
try:
    from google.colab import drive
    print("Requesting Google Drive mount (click 'Connect to Google Drive' if prompted)...")
    drive.mount('/content/drive', force_remount=False)
    drive_mounted = True
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"\n[NOTICE] Google Drive mount was skipped or unavailable: {e}")
    print("Proceeding with direct browser download (no Google Drive required).\n")

if drive_mounted:
    drive_backup_dir = '/content/drive/MyDrive/road_damage_yolo_backup'
    os.makedirs(drive_backup_dir, exist_ok=True)
    for w_name in ['best.pt', 'last.pt']:
        src_w = os.path.join(train_run_dir, 'weights', w_name)
        if os.path.exists(src_w):
            shutil.copy(src_w, os.path.join(drive_backup_dir, w_name))
            print(f"Backed up {w_name} to Google Drive: {drive_backup_dir}")
    for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
        src_p = os.path.join(train_run_dir, plot)
        if os.path.exists(src_p):
            shutil.copy(src_p, os.path.join(drive_backup_dir, plot))
            print(f"Backed up plot {plot} to Google Drive")
    if os.path.exists('outputs/metrics.json'):
        shutil.copy('outputs/metrics.json', os.path.join(drive_backup_dir, 'metrics.json'))
        print("Backed up metrics.json to Google Drive")

# --- Option B: Create Zip Bundle & Trigger Direct Browser Download ---
zip_filename = 'road_damage_results.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    if os.path.exists(best_weight):
        zipf.write(best_weight, arcname='best.pt')
        print(f"Added {best_weight} to {zip_filename}")
    for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
        src_p = os.path.join(train_run_dir, plot)
        if os.path.exists(src_p):
            zipf.write(src_p, arcname=plot)
            print(f"Added {plot} to {zip_filename}")
    if os.path.exists('outputs/metrics.json'):
        zipf.write('outputs/metrics.json', arcname='metrics.json')
        print(f"Added metrics.json to {zip_filename}")

print(f"\nCreated archive: {zip_filename}")
try:
    from google.colab import files
    print("Triggering browser download for road_damage_results.zip...")
    files.download(zip_filename)
except Exception as dl_err:
    print(f"Download trigger note: {dl_err}")
    print(f"You can also right-click '{zip_filename}' in Colab's left file pane and click 'Download'.")

## 7. Smoke Test: Run Image and Video Inference

Run inference smoke tests using `src/predict_image.py` and `src/predict_video.py` to verify qualitative detections on images and video frames.
- **Image Test**: Evaluates on test images (or valid images if test split is omitted), saves annotations to `outputs/images/`, and logs coordinates to `outputs/detections.csv`.
- **Video Test**: Uses a sample video or generates a quick demo clip from test images, applies `--skip 2`, and writes `outputs/result.mp4`.

In [ ]:
import os
import glob
from pathlib import Path
import cv2

# Locate best checkpoint
BEST_WEIGHTS = "runs/train/road_damage_yolo11s/weights/best.pt"
if not os.path.exists(BEST_WEIGHTS) and os.path.exists("weights/best.pt"):
    BEST_WEIGHTS = "weights/best.pt"

# Auto-discover image directory (check test, then valid, then val)
eval_img_dir = "datasets/road_damage/test/images"
if not os.path.exists(eval_img_dir):
    eval_img_dir = "datasets/road_damage/valid/images"
if not os.path.exists(eval_img_dir):
    eval_img_dir = "datasets/road_damage/val/images"

# --- 1. Image Inference Smoke Test ---
print(f"Running Image Inference Smoke Test on '{eval_img_dir}'...")
!python src/predict_image.py \
    {eval_img_dir} \
    --weights {BEST_WEIGHTS} \
    --conf 0.25 \
    --out outputs/images \
    --csv outputs/detections.csv

# --- 2. Video Inference Smoke Test ---
sample_video = "sample_road.mp4"

# If an external video is not provided, create a short demo video from available images
if not os.path.exists(sample_video):
    test_images = sorted(glob.glob(f"{eval_img_dir}/*.jpg") + glob.glob(f"{eval_img_dir}/*.png"))[:60]
    if test_images:
        print("Creating synthetic 3-second demo video from frames for video smoke test...")
        ref_img = cv2.imread(test_images[0])
        h, w = ref_img.shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        demo_writer = cv2.VideoWriter(sample_video, fourcc, 20.0, (w, h))
        for p in test_images:
            frame = cv2.imread(p)
            if frame is not None and frame.shape[:2] == (h, w):
                demo_writer.write(frame)
        demo_writer.release()
        print(f"Generated demo video: {sample_video}")

if os.path.exists(sample_video):
    print("\nRunning Video Inference Smoke Test...")
    !python src/predict_video.py \
        {sample_video} \
        --weights {BEST_WEIGHTS} \
        --conf 0.25 \
        --skip 2 \
        --out outputs/result.mp4